# Simulating P vs. G

:::{sidebar} Discovering Game Theory
This course uses an **online textbook**: [Game Theory, a Discovery Approach](https://nordstrommath.com/IntroGameTheory2e/intro_game_theory.html), by Jennifer Nordstrom.
:::

Following on from the previous section, we would like to be able to construct a way to simulate the **Police vs. Guerrillas** game. The functions below will create the correct game matrix and will calculate the winning percentages of the various strategies. 

Based upon the winning percentages, the simulation will choose from the strategies that have performed best in the past. Using an **exponential moving average (ema)**, the winning percentages will be updated after each round of simulation. The convergence to the correct strategy mixture for each player will be sped up as the **ema** will emphasize recent results over the older, less accurate results.

The simulation functions are described below.

In [2]:
## Do not change this cell, only execute it. 
## This cell initializes Python so that pandas, numpy and scipy packages are ready to use.

from datascience import *
import random
from IPython.display import display, Markdown
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import scipy as sp
import math
import scipy.stats as stats

## Creating the Game Matrix
When playing Police vs. Guerrillas, we would like functions that create the correct game matrix given any values of

$$2\leq p,q \leq 20$$ 

where, as discussed [above](Z8.ipynb), we have:

$$g < p < 2g $$

We will initialize the values we need:
- $p$ : number of police units
- $g$ : number of guerrilla units

The player dictionaries will be created based upon the specific values of $p$ and $g$.

We will create 2 functions to accomplish the task:

1. **create_players_dict()**
2. **game_matrix()**

### Dictionaries

The function below creates player dictionaries and keeps track of the following:
- Row-Column indexes in the results table,
- Strategy numbers (integers), and
- Strategy names.

In [81]:
def create_player_dict(name, anchor_val):
    strat = np.arange(math.ceil(anchor_val / 2), anchor_val + 1)
    index = np.arange(len(strat))
    strat_name = [f"{x} - {anchor_val - x}" for x in strat]
    return {
        'player': name,
        'index': index,
        'strat': strat,
        'strat_name': strat_name
    }

We will initialize the values for

- $p$
- $g$

Then we will create both player dictionaries.

In [83]:
p = 8
g = 7
# Creating the dictionaries
P = create_player_dict('P', p)
G = create_player_dict('G', g)

To domonstrate the output, we print the results of the work above.

In [86]:
print(P)
print(G)

{'player': 'P', 'index': array([0, 1, 2, 3, 4]), 'strat': array([4, 5, 6, 7, 8]), 'strat_name': ['4 - 4', '5 - 3', '6 - 2', '7 - 1', '8 - 0']}
{'player': 'G', 'index': array([0, 1, 2, 3]), 'strat': array([4, 5, 6, 7]), 'strat_name': ['4 - 3', '5 - 2', '6 - 1', '7 - 0']}


### Game Matrix

Now that we have the player dictionaries as well as the values $p$ and $g$ initialized, the following function will create a game matrix with outcomes all set to $0.0$.

In [87]:
def game_matrix(player1, player2):
   # Create an empty DataFrame with specified index and columns
   results = pd.DataFrame(0.0, index=player1['strat_name'], columns=player2['strat_name'])
   return results

In [108]:
results = game_matrix(P,G)
results

,4 - 3,5 - 2,6 - 1,7 - 0
4 - 4,0.0,0.0,0.0,0.0
5 - 3,0.0,0.0,0.0,0.0
6 - 2,0.0,0.0,0.0,0.0
7 - 1,0.0,0.0,0.0,0.0
8 - 0,0.0,0.0,0.0,0.0


We can access the $(i,j)$ entry in the matrix above using the pandas method 
<center><span style='color:blue'><b>.iloc[i,j]</b></span></center>

In [109]:
results.iloc[3,2]

0.0

### Init() Function

The interior entries of the **results** dataframe will be winning percentages of the row and column strategies when played against one another.

To initialize these values, we begin by playing each column strategy against each row strategy. We plan for 40 simulated battles with $10$ **ghost results**.  With $5$ ghost wins and $5$ ghost losses, the win percentages will skew toward the center (e.g. 0.5). The **ghost results** approach is typical when we wish to avoid seeding the simulation with extreme values, in this case, zeroes and ones.

With $10$ ghost results included, the simulation results appear as though $50$ total results were simulated. The **init()** function code is given below.

In [110]:
def init(p_max, g_max, p_strat, g_strat):
   A_0 = p_strat ; B_0 = p_max - A_0
   A = np.random.choice([A_0, B_0])
   B = p_max - A ;
   res = ()
   for k in range(40):
      a_0 = g_strat ; b_0 = g_max - a_0
      a = np.random.choice([a_0, b_0])
      b = g_max - a
      if ( ( a > A ) or ( b > B ) ):
         v = 0
      else:
         v = 1
      res = np.append(res, v)
   temp = ( sum(res) + 5 ) / 50
   t = round(temp,3)
   ## uncomment line below to help with debugging the init() function.
   #   print(f"police play ({A},{B}) while guerrillas play ({a},{b})")
   return t       

The function above will be used in a nested for-loop to calculate the needed values and assign them to the correct entry in the **results** dataframe. The code below accomplishes the initialization of the **results** table created above.

In [118]:
for i in P['index']:
   print(f"Processing row {i}...") # Debugging line
   for j in G['index']:
      results.iloc[i, j] = init(p,g, P['strat'][i],G['strat'][j])

results

Processing row 0...
Processing row 1...
Processing row 2...
Processing row 3...
Processing row 4...


,4 - 3,5 - 2,6 - 1,7 - 0
4 - 4,0.9,0.10,0.10,0.10
5 - 3,0.4,0.44,0.10,0.10
6 - 2,0.1,0.52,0.54,0.10
7 - 1,0.1,0.10,0.46,0.50
8 - 0,0.1,0.10,0.10,0.48
